# 检索攻击效果评估（ASR-R + Precision@K）

本 Notebook 实现对 RACG 检索投毒攻击的评估流程：

1. **数据库投毒**：将经过检索对抗优化的有毒 BFP 文档插入 APR 知识库
2. **执行检索**：根据 `apr/bfp.yml` 中的 `retrieval_strategy` 执行 sparse / dense / hybrid；当策略为 `all` 时同时生成三类检索结果
3. **指标评估**：同步计算 ASR-R（Top-K 中至少命中一篇有毒文档的查询比例）与 ImportSnare 风格的未归一化 Precision@K（每个查询的 Top-K 中平均包含的有毒文档数量）

评估行为由 `configs/evaluation/retrieval.yml` 的以下开关控制：

- `eval_mode`：`per_cwe` 仅逐 CWE 独立评估；`mixed` 仅评估合并后的 `CWE-MIXED`；`both` 先逐 CWE、再追加 `CWE-MIXED`。
  - **数据按 CWE 划分（v1/v4）**：`eval_cwe` 表示参与独立评估或合并评估的源 CWE 列表。
  - **数据为 CWE-MIXED（v2/v3，`eval_cwe=["CWE-MIXED"]`）**：任何配置模式均自动降级为单次 `mixed` 评估并打印警告。
- `eval_retrieval_naive`：`True` 时一次运行自动完成「naive 注入-评估-清除-poisoned 注入-评估-清除」；`False` 时仅产出 poisoned 结果。naive 文档的 buggy/fixed 均取干净 entity 版本，但仍标记 `is_poisoned=True`。
- `eval_unfinished_jailbreak_target`：仅影响 poisoned 模式。`True` 时，已有 `poisoned_buggy_code` 但没有 `jailbreak_code` 的目标使用 `poisoned_buggy_code + entity.fixed_code` 注入；`False` 时保持严格报错。该模式只用于检索侧 ASR-R/Precision@K，不应直接用于生成攻击评估。

结果文件与日志路径均携带 `division / embedder / optimizer / label / CWE / alpha / N(投毒目标数) / M(测试样本数)` 等信息，检索评估与生成评估共用同一命名约定（详见后续 Cell）。

In [ ]:
# HF_ENDPOINT 须在首次 import huggingface_hub 前设置；若曾先跑过含 HF 的 import，请 Kernel → Restart 后从本格重跑。
import sys
import os
from pathlib import Path
# Honor any HF_ENDPOINT selected by the caller.
print("HF_ENDPOINT =", os.environ.get("HF_ENDPOINT"))
# 向上查找项目根目录（含 configs/ 目录的最近祖先）
PROJECT_ROOT = Path(os.path.abspath("")).resolve()
while not (PROJECT_ROOT / "configs").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

from src.utils.io import load_json, save_json, load_yaml
from src.utils.log import setup_logging
from src.models.retriever.Base import create_embedder
from src.rag.bfp.milvus_client import BFPMilvusClient
from src.rag.bfp.retriever import BFPRetriever
from src.evaluation.retrieval.metrics import asr_r

setup_logging()

## 1. 加载配置

In [ ]:
from datetime import datetime

# 单次 notebook 运行复用同一个分钟级时间戳。
RUN_TIMESTAMP = datetime.now().strftime("%Y-%m-%d-%H-%M")
eval_config = load_yaml(str(PROJECT_ROOT / "configs/evaluation/retrieval.yml"))
db_config = load_yaml(str(PROJECT_ROOT / eval_config["database"]["milvus_config"]))
model_config = load_yaml(str(PROJECT_ROOT / db_config["database"]["model"]))

# ── 数据路径 ──────────────────────────────────────────────────────────────────
poisoned_file_path = str(PROJECT_ROOT / eval_config["data"]["poisoned_file_path"])
test_file_path = str(PROJECT_ROOT / eval_config["data"]["test_file_path"])

# ── 评估参数 ──────────────────────────────────────────────────────────────────
VALID_EVAL_MODES = {"per_cwe", "mixed", "both"}


def resolve_eval_groups(eval_cfg):
    """返回 (eval_cwe, 配置模式, 生效模式, [(slug, source_cwes), ...])。"""
    if "eval_isolated" in eval_cfg:
        raise ValueError(
            "eval.eval_isolated 已移除；请使用 eval.eval_mode: per_cwe | mixed | both"
        )
    raw_cwe = eval_cfg.get("eval_cwe")
    if not isinstance(raw_cwe, (list, tuple)) or isinstance(raw_cwe, (str, bytes)):
        raise TypeError("eval.eval_cwe 必须是非空 CWE 字符串数组")
    if not raw_cwe:
        raise ValueError("eval.eval_cwe 不能为空")
    if any(not isinstance(cwe, str) or not cwe.strip() for cwe in raw_cwe):
        raise TypeError("eval.eval_cwe 中每一项都必须是非空字符串")
    cwe_list = list(dict.fromkeys(cwe.strip() for cwe in raw_cwe))

    configured_mode = str(eval_cfg.get("eval_mode", "")).strip().lower()
    if configured_mode not in VALID_EVAL_MODES:
        raise ValueError(
            "eval.eval_mode 必须是 per_cwe、mixed 或 both，"
            f"实际为 {eval_cfg.get('eval_mode')!r}"
        )
    if "CWE-MIXED" in cwe_list and cwe_list != ["CWE-MIXED"]:
        raise ValueError("eval.eval_cwe 不能同时包含 CWE-MIXED 与普通 CWE")

    effective_mode = "mixed" if cwe_list == ["CWE-MIXED"] else configured_mode
    per_cwe_groups = [(cwe, [cwe]) for cwe in cwe_list]
    mixed_group = [("CWE-MIXED", list(cwe_list))]
    if effective_mode == "per_cwe":
        groups = per_cwe_groups
    elif effective_mode == "mixed":
        groups = mixed_group
    else:
        groups = per_cwe_groups + mixed_group
    return cwe_list, configured_mode, effective_mode, groups


eval_cfg = eval_config.get("eval", {})
eval_cwe, configured_eval_mode, effective_eval_mode, eval_groups = resolve_eval_groups(eval_cfg)
if configured_eval_mode != effective_eval_mode:
    print(
        f"[WARN] eval_cwe={eval_cwe} 已是混合数据；eval_mode={configured_eval_mode} "
        "自动降级为 effective_eval_mode=mixed"
    )
# eval_top_k 支持标量或数组（数组时对每个 Top-K 分别计算 ASR-R 与 Precision@K），此处统一规范为列表
_eval_top_k_raw = eval_config.get("eval", {}).get("eval_top_k", 10)
eval_top_k = list(_eval_top_k_raw) if isinstance(_eval_top_k_raw, (list, tuple)) else [_eval_top_k_raw]
eval_retrieval_naive = bool(eval_config.get("eval", {}).get("eval_retrieval_naive", False))
eval_unpoisoned_target = bool(eval_config.get("eval", {}).get("eval_unpoisoned_target", False))
eval_unfinished_jailbreak_target = bool(
    eval_config.get("eval", {}).get("eval_unfinished_jailbreak_target", False)
)
retrieval_cfg = db_config.get("retrieval", {})
retrieval_strategy = retrieval_cfg.get("retrieval_strategy", "hybrid").lower()
query_target = retrieval_cfg.get("query_target", "buggy_code")
alpha_weight = retrieval_cfg.get("alpha_weight", 0.5)

print(f"poisoned_file_path : {poisoned_file_path}")
print(f"test_file_path     : {test_file_path}")
print(f"eval_cwe           : {eval_cwe}")
print(f"eval_top_k         : {eval_top_k}  # ASR-R 与 Precision@K 评估的 Top-K 列表")
print(f"eval_mode          : {configured_eval_mode}")
print(f"effective_eval_mode: {effective_eval_mode}")
print(f"eval_groups        : {[slug for slug, _ in eval_groups]}")
print(f"eval_naive         : {eval_retrieval_naive}  # True=自动产出 naive 与 poisoned 两套结果")
print(f"eval_unpoisoned_target : {eval_unpoisoned_target}  # True=[poisoned]模式同时插入未完成攻击的目标，保证与naive文档总量一致")
print(f"eval_unfinished_jailbreak_target : {eval_unfinished_jailbreak_target}  # True=允许攻击buggy + 干净fixed的检索侧专用回退")
print(f"retrieval_strategy : {retrieval_strategy}")
print(f"query_target       : {query_target}")
print(f"alpha_weight       : {alpha_weight}")

## 2. 解析 CWE 信息 & 初始化模型与数据库

In [ ]:
# 数据划分类型（gray/black/white）从 poisoned_file_path 路径解析（.../data/query/{division}/...）
# optimizer_name 在 Cell 7 加载投毒文件后从 retrieval_attack.configs.attack.optimizer 解析
# cwe_slug 不在此处提前计算，改为驱动循环内按当前 CWE 分组动态生成（见后续驱动 Cell）
def _derive_division(path):
    """从形如 .../data/query/{division}/... 的路径中解析数据划分类型。"""
    parts = Path(path).parts
    if "query" in parts:
        i = parts.index("query")
        if i + 1 < len(parts):
            return parts[i + 1]
    return "gray"


division = _derive_division(poisoned_file_path)
print(f"eval_cwe       : {eval_cwe}")
print(f"eval_mode      : {configured_eval_mode}")
print(f"effective_mode : {effective_eval_mode}")
print(f"division       : {division}")

# 初始化嵌入模型
embedder = create_embedder(model_config)
embedder_name = embedder.get_model_name(with_provider=False)
test_emb = embedder.embed_queries(["int main() { return 0; }"])
print(f"\nEmbedder ready | dim={test_emb.shape[1]} | embedder={embedder_name}")

# 初始化 Milvus 客户端与检索器
if not os.path.isabs(db_config["database"]["uri"]):
    db_config["database"]["uri"] = str(PROJECT_ROOT / db_config["database"]["uri"])
milvus_client = BFPMilvusClient(db_config)
retriever = BFPRetriever(embedder, milvus_client)
print(f"Milvus connected | collection={db_config.get('database', {}).get('collection_name', '')} | uri={milvus_client.uri}")


## 3. 数据库投毒（函数定义）

将投毒、载入测试集、检索、保存等步骤封装为可复用函数，供后续驱动循环按「CWE 分组 × naive/poisoned 模式」反复调用。

- `make_poison_docs(...)`：把投毒条目组织为 Milvus 文档列表。naive 模式的 buggy/fixed 均取干净 `entity` 版本；poisoned 模式使用 `poisoned_buggy_code + jailbreak_code`，或在显式允许未完成 jailbreak 时使用 `poisoned_buggy_code + entity.fixed_code`。所有注入目标均保留 `is_poisoned=True`。
- `inject_poison(...)`：先清除已有投毒文档，再按 `cwe_list` 抽取并插入 BFP 知识库。`eval_unpoisoned_target` 与 `eval_unfinished_jailbreak_target` 分别控制“检索攻击未完成”和“仅 jailbreak 未完成”的目标，二者互不替代。合并文件中的 `entity.id` 必须全局唯一，notebook 不再运行期重命名。

In [ ]:
from src.utils.model import truncate_text
from src.utils.io import normalize_bfp_record_id, validate_global_entity_ids

# 新契约要求 Section 3 已完成跨 CWE ID 唯一化；检索阶段不再临时哈希。
poisoned_data = load_json(poisoned_file_path)
validate_global_entity_ids(poisoned_data)
print(f"投毒文件 CWE 类别: {list(poisoned_data.keys())}")


def _derive_optimizer_name(data):
    for entries in data.values():
        for entry in entries or []:
            opt = (((entry.get("retrieval_attack") or {}).get("configs") or {}).get("attack") or {}).get("optimizer")
            if opt:
                return str(opt).lower()
    return "unknown"


optimizer_name = _derive_optimizer_name(poisoned_data)
print(f"optimizer_name : {optimizer_name}")


def _is_unfinished_jailbreak_target(entry):
    """Return True only when retrieval attack is done but jailbreak is unfinished."""
    attacked_buggy = (entry.get("retrieval_attack") or {}).get("poisoned_buggy_code")
    jailbreak_code = (
        ((entry.get("generation_attack") or {}).get("jailbreak_attack") or {}).get("jailbreak_code")
    )
    return bool(attacked_buggy) and not bool(jailbreak_code)


def select_injection_codes(
    entry,
    *,
    use_naive,
    allow_clean_poisoned=False,
    allow_unfinished_jailbreak=False,
):
    """Select the exact code pair inserted for a retrieval-evaluation document."""
    entity = entry.get("entity") or {}
    clean_buggy = entity.get("buggy_code") or ""
    clean_fixed = entity.get("fixed_code") or ""
    if use_naive:
        if not clean_buggy or not clean_fixed:
            raise ValueError("naive injection requires clean entity.buggy_code/fixed_code")
        return clean_buggy, clean_fixed

    attacked_buggy = (entry.get("retrieval_attack") or {}).get("poisoned_buggy_code") or ""
    if not attacked_buggy:
        if allow_clean_poisoned and clean_buggy and clean_fixed:
            return clean_buggy, clean_fixed
        raise ValueError("poisoned injection requires retrieval_attack.poisoned_buggy_code")

    jailbreak_code = (
        ((entry.get("generation_attack") or {}).get("jailbreak_attack") or {}).get("jailbreak_code")
        or ""
    )
    if not jailbreak_code:
        if allow_unfinished_jailbreak:
            if not clean_fixed:
                raise ValueError(
                    "unfinished-jailbreak fallback requires entity.fixed_code"
                )
            return attacked_buggy, clean_fixed
        raise ValueError("poisoned injection requires generation_attack.jailbreak_attack.jailbreak_code")
    return attacked_buggy, jailbreak_code


def collect_poisoned_entries(
    cwe_list,
    use_naive=False,
    include_unpoisoned=False,
    include_unfinished_jailbreak=False,
    verbose=False,
):
    """Collect globally unique poisoned targets for naive or poisoned injection."""
    entries = []
    seen_ids = set()
    for cwe in cwe_list:
        raw_count = 0
        kept_count = 0
        unfinished_fallback_count = 0
        for entry in poisoned_data.get(cwe, []):
            entity = entry.get("entity") or {}
            has_attack = bool((entry.get("retrieval_attack") or {}).get("poisoned_buggy_code"))
            if use_naive:
                if not entity.get("buggy_code") or not entity.get("fixed_code"):
                    continue
            elif not has_attack and not (include_unpoisoned and entity.get("buggy_code") and entity.get("fixed_code")):
                continue
            elif _is_unfinished_jailbreak_target(entry):
                if not include_unfinished_jailbreak:
                    raise ValueError(f"[{cwe}] entity.id={entity.get('id')!r} 已有 poisoned_buggy_code 但缺少 jailbreak_code")
                if not entity.get("fixed_code"):
                    raise ValueError(
                        f"[{cwe}] entity.id={entity.get('id')!r} 未完成 jailbreak，"
                        "但缺少回退所需的 entity.fixed_code"
                    )
                unfinished_fallback_count += 1
            raw_count += 1
            norm_id = normalize_bfp_record_id(entity.get("id"))
            if norm_id is None:
                raise ValueError(f"[{cwe}] 投毒样本缺少 entity.id")
            if norm_id in seen_ids:
                raise ValueError(f"投毒文件仍含重复 entity.id={norm_id!r}；请重跑 distributed_io Section 3")
            seen_ids.add(norm_id)
            entries.append(entry)
            kept_count += 1
        if verbose:
            print(
                f"  [{cwe}] {raw_count} 条（唯一 ID 保留 {kept_count} 条，"
                f"未完成 jailbreak 回退 {unfinished_fallback_count} 条）"
            )
    return entries


def make_poison_docs(
    entries,
    use_naive,
    include_unpoisoned=False,
    include_unfinished_jailbreak=False,
):
    """Build Milvus docs under the configured clean/attacked/fallback semantics."""
    if not entries:
        return []
    code_pairs = [
        select_injection_codes(
            entry,
            use_naive=use_naive,
            allow_clean_poisoned=include_unpoisoned,
            allow_unfinished_jailbreak=include_unfinished_jailbreak,
        )
        for entry in entries
    ]
    embed_texts = [pair[0] for pair in code_pairs]
    print(f"  [{'naive' if use_naive else 'poisoned'}] 正在生成 {len(embed_texts)} 条 dense_vector")
    embeddings = embedder.embed_documents(embed_texts)
    docs = []
    for i, entry in enumerate(entries):
        entity = entry.get("entity") or {}
        try:
            doc_id = int(entity.get("id"))
        except (TypeError, ValueError) as exc:
            raise ValueError(f"Milvus doc_id 必须为 INT64: {entity.get('id')!r}") from exc
        fixed_code = code_pairs[i][1]
        docs.append({
            "doc_id": doc_id, "chunk_id": 0, "total_chunks": 1,
            "buggy_code": truncate_text(embed_texts[i]),
            "fixed_code": truncate_text(fixed_code),
            "is_poisoned": True, "source": entity.get("source", ""),
            "dense_vector": embeddings[i].tolist(),
        })
    return docs


def inject_poison(
    cwe_list,
    use_naive,
    include_unpoisoned=False,
    include_unfinished_jailbreak=False,
):
    entries = collect_poisoned_entries(
        cwe_list,
        use_naive,
        include_unpoisoned,
        include_unfinished_jailbreak,
    )
    print(f"待注入文档数量: {len(entries)}（cwe={cwe_list}, naive={use_naive}）")
    cleared = milvus_client.clear_poisoned_docs()
    print(f"  注入前已清除投毒文档: {cleared} 条")
    docs = make_poison_docs(
        entries, use_naive, include_unpoisoned, include_unfinished_jailbreak
    )
    if not docs:
        print("  ⚠ 无可注入的投毒文档，跳过插入")
        return 0

    inserted = milvus_client.insert_documents(docs)
    print(f"  成功插入投毒文档: {inserted} 条")
    return inserted

## 4. 加载测试查询集（函数定义）

`load_test_queries(cwe_list)`：按 `cwe_list` 从测试查询集（嵌套 dict 格式）展平为查询列表。

In [ ]:
# 测试查询集也是嵌套 dict 格式: {CWE: [{id, buggy_code, ...}, ...]}
test_data_raw = load_json(test_file_path)
print(f"测试查询集 CWE 类别: {list(test_data_raw.keys())}")


def _query_id(query):
    entity = query.get("entity")
    return entity.get("id") if isinstance(entity, dict) else query.get("id")


def load_test_queries(cwe_list):
    """按 cwe_list 展平查询，并按规范化的查询 ID 跨 CWE 去重。"""
    queries = []
    seen_ids = set()
    for cwe in cwe_list:
        cwe_queries = test_data_raw.get(cwe, [])
        kept_count = 0
        for query in cwe_queries:
            normalized_id = normalize_bfp_record_id(_query_id(query))
            if normalized_id is not None and normalized_id in seen_ids:
                continue
            if normalized_id is not None:
                seen_ids.add(normalized_id)
            queries.append(query)
            kept_count += 1
        print(f"  [{cwe}] {len(cwe_queries)} 条（去重后保留 {kept_count} 条）")
    print(f"  测试查询总量: {len(queries)}（cwe={cwe_list}）")
    return queries

## 5. 执行检索（函数定义）

`run_retrieval(test_data)`：按 `retrieval_strategy` 执行 sparse / dense / hybrid；策略为 `all` 时三类全跑。返回 `{strategy: results}`。

In [ ]:
retrieval_top_k = int(retrieval_cfg.get("top_k", 10))
if max(eval_top_k) > retrieval_top_k:
    raise ValueError(
        f"APR 数据库检索深度 top_k={retrieval_top_k} 小于 eval_top_k 的最大值 {max(eval_top_k)}。"
        "请先提高 configs/database/bfp/apr.yml 中的 retrieval.top_k。"
    )
valid_strategies = {"dense", "sparse", "hybrid", "all"}
if retrieval_strategy not in valid_strategies:
    raise ValueError(
        f"Unsupported retrieval_strategy={retrieval_strategy!r}; "
        f"expected one of {sorted(valid_strategies)}"
    )

strategy_order = ["sparse", "dense", "hybrid"] if retrieval_strategy == "all" else [retrieval_strategy]


def run_retrieval(test_data):
    """按 strategy_order 执行检索，返回 {strategy: results}。test_data 为空时返回空结果。"""
    results_by_strategy = {}
    if not test_data:
        print("  ⚠ 测试查询为空，跳过检索")
        return {strategy: [] for strategy in strategy_order}

    for strategy in strategy_order:
        if strategy == "sparse":
            print("=== 稀疏检索（Sparse Retrieval / BM25）===")
            results = retriever.sparse_retrieve(
                test_data,
                top_k=retrieval_top_k,
                query_target=query_target,
            )
        elif strategy == "dense":
            print("=== 密集检索（Dense Retrieval）===")
            results = retriever.dense_retrieve(
                test_data,
                top_k=retrieval_top_k,
                query_target=query_target,
            )
        else:
            print(f"=== 混合检索（Hybrid Retrieval, alpha={alpha_weight}）===")
            results = retriever.hybrid_retrieve(
                test_data,
                top_k=retrieval_top_k,
                alpha=alpha_weight,
                query_target=query_target,
            )

        results_by_strategy[strategy] = results
        print(f"完成 {strategy} 检索，共 {len(results)} 条结果")
        if results and results[0]["retrieval_results"]:
            top_hit = results[0]["retrieval_results"][0]
            print(
                f"示例 Top-1: doc_id={top_hit['doc_id']}, "
                f"is_poisoned={top_hit['is_poisoned']}, score={top_hit['score']:.4f}"
            )
        print()
    return results_by_strategy

## 6. 保存检索结果（函数定义）

`save_retrieval_results(results_by_strategy, cwe_slug, label, n_target, n_test)`：按新版模板写入结果文件，返回 `{strategy: path}`。

路径模板：
```
data/query/{division}/test/results/{embedder}/{optimizer}-{timestamp}/{label}/
  {cwe_slug}_alpha_{alpha_label}_target_{N}_test_{M}_retrieval_results.json
```
其中 `alpha_label` 命名约定为 sparse→0、dense→1、hybrid→{alpha_weight}；`N`=当前分组投毒目标文档数，`M`=当前分组测试样本数。

In [ ]:
# 输出路径:
#   data/query/{division}/test/results/{embedder}/{optimizer}-{timestamp}/{label}/
#     {cwe_slug}_alpha_{alpha_label}_target_{N}_test_{M}_retrieval_results.json
# 命名约定：sparse → alpha_0，dense → alpha_1，hybrid → alpha_{alpha_weight}
alpha_label_by_strategy = {
    "sparse": "0",
    "dense": "1",
    "hybrid": str(alpha_weight),
}
display_name_by_strategy = {
    "sparse": "稀疏检索（Sparse）",
    "dense": "密集检索（Dense）",
    "hybrid": "混合检索（Hybrid）",
}


def save_retrieval_results(results_by_strategy, cwe_slug, label, n_target, n_test):
    """按新版路径保存各策略检索结果，返回 {strategy: path}。

    n_target: 当前分组投毒目标文档数量；n_test: 当前分组测试样本数量。
    """
    results_base = (
        PROJECT_ROOT / "data/query" / division / "test/results"
        / embedder_name / f"{optimizer_name}-{RUN_TIMESTAMP}" / label
    )

    output_paths = {}
    for strategy, results in results_by_strategy.items():
        alpha_label = alpha_label_by_strategy[strategy]
        fname = (
            f"{cwe_slug}_alpha_{alpha_label}"
            f"_target_{n_target}_test_{n_test}_retrieval_results.json"
        )
        output_path = str(results_base / fname)
        save_json(results, output_path)
        output_paths[strategy] = output_path
        print(f"  {display_name_by_strategy[strategy]}结果已保存: {output_path}")
    return output_paths

## 7. 驱动循环：注入 → 检索 → 保存 → 清除

按「CWE 分组 × 模式」驱动整个评估流程，结果路径收集到 `all_output_paths`，每组每模式的 `(N, M)` 计数收集到 `group_counts`（结构 `{cwe_slug: {label: (N, M)}}`）：

- 分组矩阵由 `eval_mode` 控制：`per_cwe` 仅逐 CWE，`mixed` 仅整体 `CWE-MIXED`，`both` 同时执行两者。
  - 合并组使用 `eval_cwe` 全部源 CWE；测试查询继续按规范化 `entity.id` 跨 CWE 去重。
  - 数据本身为 CWE-MIXED（v2/v3）时自动降级为一次整体评估。
- `eval_retrieval_naive=True`：每个分组依次完成 naive 与 poisoned 两轮「注入-评估-清除」；`False` 时仅产出 poisoned。
- **投毒目标文档数 N 按模式独立统计**：naive 目标可使用纯净 entity 代码；poisoned 默认要求已完成检索对抗优化。
- `eval_unpoisoned_target=True`：poisoned 模式以原始 `buggy_code/fixed_code` 补齐未完成检索攻击的目标，使文档总量与 naive 一致。
- `eval_unfinished_jailbreak_target=True`：poisoned 模式允许已完成检索攻击但未完成 jailbreak 的目标，以 `poisoned_buggy_code + entity.fixed_code` 注入；这些结果仅用于检索指标。

In [ ]:
# ── 复用配置单元格已验证的评估分组 (cwe_slug, source_cwes) ────────────────────
# mixed/both 的整体组保留各 CWE 下的独立投毒文档；测试查询由 load_test_queries 跨 CWE 去重。
groups = list(eval_groups)

# (use_naive, label) 列表；naive=True 表示原始无毒文档，False 表示有毒攻击文档
label_modes = [(True, "naive"), (False, "poisoned")] if eval_retrieval_naive else [(False, "poisoned")]

print(f"评估分组   : {[slug for slug, _ in groups]}")
print(f"评估模式   : {[label for _, label in label_modes]}")

# 收集结构: {cwe_slug: {label: {strategy: path}}}
all_output_paths = {}
# 每组的 (N=投毒目标文档数, M=测试样本数)，供路径命名与日志复用
group_counts = {}
# 每组/模式的注入语义统计，供详细日志与 overview.md 复用
group_injection_stats = {}

for cwe_slug, group in groups:
    all_output_paths.setdefault(cwe_slug, {})
    print(f"\n{'#'*70}")
    print(f"#  CWE 分组: {cwe_slug}  (cwe_list={group})")
    print(f"{'#'*70}")

    # 该分组的测试查询只需载入一次，naive / poisoned 共用
    group_test_data = load_test_queries(group)
    n_test = len(group_test_data)
    group_counts.setdefault(cwe_slug, {})
    group_injection_stats.setdefault(cwe_slug, {})
    print(f"  测试样本数 M={n_test}")

    for use_naive, label in label_modes:
        # include_unpoisoned 仅在 poisoned 模式下生效：eval_unpoisoned_target=True 时，
        # 将尚未完成检索攻击的目标一并以原始 buggy_code 补齐注入，使 [poisoned] 模式的文档
        # 总量与 [naive] 模式一致，避免两种模式下知识库竞争文档数量不同导致 ASR-R 对比失真。
        include_unpoisoned = eval_unpoisoned_target and not use_naive
        # include_unfinished_jailbreak 也仅在 poisoned 模式生效：保留 attacked buggy，
        # fixed_code 回退到 entity.fixed_code；该结果只用于检索侧指标。
        include_unfinished_jailbreak = (
            eval_unfinished_jailbreak_target and not use_naive
        )

        # N（投毒目标文档数）按模式独立统计：
        #   naive 仅需原始 entity.buggy_code/fixed_code（对抗优化尚未完成时也可评估）；
        #   poisoned 默认需已完成检索对抗优化（含 poisoned_buggy_code）；
        #   若 include_unpoisoned=True，则 poisoned 模式的 N 会与 naive 一致。
        #   verbose=True 打印逐 CWE 的（去重前/CWE 内去重后）条目数，便于核对 N 的构成。
        selected_entries = collect_poisoned_entries(
            group,
            use_naive=use_naive,
            include_unpoisoned=include_unpoisoned,
            include_unfinished_jailbreak=include_unfinished_jailbreak,
            verbose=True,
        )
        n_target = len(selected_entries)
        unfinished_fallback_count = sum(
            1 for entry in selected_entries
            if (not use_naive and _is_unfinished_jailbreak_target(entry))
        )
        group_counts[cwe_slug][label] = (n_target, n_test)
        group_injection_stats[cwe_slug][label] = {
            "target_count": n_target,
            "unfinished_jailbreak_fallback_count": unfinished_fallback_count,
        }
        print(f"\n{'='*70}")
        print(
            f"  [{cwe_slug}] 模式: {label}（naive={use_naive}, "
            f"include_unpoisoned={include_unpoisoned}, "
            f"include_unfinished_jailbreak={include_unfinished_jailbreak}）"
            f"| 投毒目标文档数 N={n_target}, 未完成 jailbreak 回退={unfinished_fallback_count}"
        )
        if unfinished_fallback_count:
            print(
                "  [WARN] 本轮包含未完成 jailbreak 的目标：fixed_code 使用 entity.fixed_code。"
                "这些结果仅用于 ASR-R/Precision@K，不应作为生成攻击评估输入。"
            )
        print(f"{'='*70}")

        # 1) 注入（内部已先清除旧投毒文档）
        inserted_count = inject_poison(
            group,
            use_naive,
            include_unpoisoned=include_unpoisoned,
            include_unfinished_jailbreak=include_unfinished_jailbreak,
        )
        if inserted_count != n_target:
            raise RuntimeError(
                f"注入数量 {inserted_count} 与 target_N={n_target} 不一致，拒绝继续评估"
            )

        # 2) 检索
        group_results = run_retrieval(group_test_data)

        # 3) 保存（路径含 N/M）
        paths = save_retrieval_results(group_results, cwe_slug, label, n_target, n_test)
        all_output_paths[cwe_slug][label] = paths

        # 4) 清除本轮投毒文档，恢复干净库以便下一轮 / 下一分组
        cleared = milvus_client.clear_poisoned_docs()
        print(f"  本轮结束已清除投毒文档: {cleared} 条")

print(f"\n{'#'*70}")
print("所有分组评估完成，结果路径汇总：")
for cwe_slug, by_label in all_output_paths.items():
    for label, paths in by_label.items():
        for strategy, path in paths.items():
            print(f"  [{cwe_slug}][{label}][{strategy}] {path}")

## 8. 计算 ASR-R 与 Precision@K 指标 + 日志持久化

遍历驱动循环收集的 `all_output_paths`（结构 `{cwe_slug: {label: {strategy: path}}}`），逐 CWE 分组打印各检索策略的 ASR-R；当同一分组同时存在 naive 与 poisoned 时并列对比。每份报告同时持久化到：
```
logs/evaluation/retrieval/{embedder}/{optimizer}-{timestamp}/{label}/top-{K}/
  {cwe_slug}_alpha_{alpha_label}_target_{N}_test_{M}_retrieval_results.txt
```
`from_existing_files` 模式只接受新契约的 `target_*_test_*` 文件（`N/M` 从文件名解析）。

In [ ]:
# =============================================================================
# 配置区：选择运行模式
# =============================================================================
import re
from collections import Counter
from datetime import datetime

MODE = "from_memory"  # 使用内存中的检索结果（需先运行前面所有单元格，复用驱动循环产出的 all_output_paths）
# MODE = "from_existing_files"  # 从已有检索结果文件直接加载（按矩阵重建路径集合，用 glob 匹配 N/M）

# 当 MODE = "from_existing_files" 时：
#   - 设为某个具体的 label 结果目录（形如 ".../{optimizer}-{timestamp}/{label}"）→ 仅加载 eval_mode 要求的分组
#   - 设为 None → 按 eval_mode/eval_retrieval_naive 自动重建当前运行的路径集合
EXISTING_RESULTS_DIR = None

# 是否打印受害查询与有毒文档的对应详情
SHOW_DETAIL = True

# 结果与日志共用 Cell 3 创建的 RUN_TIMESTAMP。

def _parse_counts_from_path(results_path):
    """从新契约文件名 ..._target_{N}_test_{M}_retrieval_results.json 中解析 (N, M)。"""
    m = re.search(r"_target_(\d+)_test_(\d+)_", Path(results_path).name)
    if m:
        return int(m.group(1)), int(m.group(2))
    return None, None


def build_retrieval_report(title, results_path, eval_top_k, show_detail=False):
    """加载检索结果文件，计算 ASR-R 与 Precision@K，打印并返回 (metrics, report_text)。"""
    data = load_json(results_path)
    metrics = asr_r(data, eval_top_k=eval_top_k)

    lines = []
    lines.append("=" * 55)
    lines.append(f"  {title}")
    lines.append("=" * 55)
    lines.append(f"  ASR-R @ Top-{eval_top_k}  : {metrics['asr_r']:.4f}  ({metrics['asr_r']*100:.2f}%)")
    lines.append(f"  Precision@K @ Top-{eval_top_k} (unnormalized): {metrics['precision_at_k']:.4f}")
    lines.append(f"  有毒文档累计出现次数 : {metrics['total_poisoned_count']}")
    lines.append(f"  Precision@K 口径      : 未除以 K，表示每个查询 Top-{eval_top_k} 中平均包含的有毒文档数，范围 [0, {eval_top_k}]")
    lines.append(f"  命中查询数           : {metrics['hit_count']} / {metrics['total_queries']}")

    all_ranks = []
    for q in metrics["per_query"]:
        all_ranks.extend(q["poisoned_ranks"])
    if all_ranks:
        rank_dist = Counter(all_ranks)
        lines.append("  有毒文档命中排名分布 :")
        for rank in sorted(rank_dist):
            lines.append(f"    Rank {rank:2d}: {rank_dist[rank]} 次")
    else:
        lines.append("  有毒文档命中排名分布 : 无命中")

    if show_detail:
        hit_queries = [q for q in metrics["per_query"] if q["hit"]]
        if hit_queries:
            lines.append("  受害查询与有毒文档：")
            for counter, q in enumerate(hit_queries, start=1):
                doc_ids = q.get("poisoned_doc_ids", [])
                ranks = q["poisoned_ranks"]
                lines.append(f"    {counter}. 受害查询：{q['id']}")
                for doc_id, rank in zip(doc_ids, ranks):
                    lines.append(f"       - 有毒文档：{doc_id}（Rank{rank:2d}）")
        else:
            lines.append("  受害查询与有毒文档：无命中")

    lines.append("=" * 55)
    report_text = "\n".join(lines)
    print("\n" + report_text)
    return metrics, report_text


def write_retrieval_log(report_text, cwe_slug, label, alpha_label, results_path, top_k):
    """将 ASR-R 与 Precision@K 报告持久化到:
    logs/evaluation/retrieval/{embedder}/{optimizer}-{date}/{label}/top-{top_k}/
      {cwe_slug}_alpha_{alpha_label}_target_{N}_test_{M}_retrieval_results.txt
    """
    n_target, n_test = _parse_counts_from_path(results_path)
    n_target = 0 if n_target is None else n_target
    n_test = 0 if n_test is None else n_test
    injection_stats = (
        globals().get("group_injection_stats", {}).get(cwe_slug, {}).get(label, {})
    )
    fallback_count = injection_stats.get("unfinished_jailbreak_fallback_count", "unknown")

    log_dir = (
        PROJECT_ROOT / "logs/evaluation/retrieval" / embedder_name
        / f"{optimizer_name}-{RUN_TIMESTAMP}" / label / f"top-{top_k}"
    )
    log_dir.mkdir(parents=True, exist_ok=True)
    fname = (
        f"{cwe_slug}_alpha_{alpha_label}"
        f"_target_{n_target}_test_{n_test}_retrieval_results.txt"
    )
    log_path = log_dir / fname

    header = [
        "Retrieval Attack Evaluation (ASR-R + Precision@K)",
        f"Timestamp : {datetime.now().isoformat(timespec='seconds')}",
        f"Embedder  : {embedder_name}",
        f"Optimizer : {optimizer_name}",
        f"Division  : {division}",
        f"CWE       : {cwe_slug}",
        f"Label     : {label}",
        f"Alpha     : {alpha_label}",
        f"Top-K     : {top_k}",
        f"Eval mode : {configured_eval_mode}",
        f"Effective eval mode : {effective_eval_mode}",
        f"Eval unfinished jailbreak target : {eval_unfinished_jailbreak_target}",
        f"Unfinished jailbreak fallbacks  : {fallback_count}",
        "Result scope: retrieval-only if unfinished jailbreak fallbacks > 0",
        f"Source    : {results_path}",
        "",
    ]
    log_path.write_text("\n".join(header) + report_text + "\n", encoding="utf-8")
    print(f"  [log] 检索攻击评估报告已保存: {log_path}")
    return str(log_path)


# =============================================================================
# 加载评估配置
# =============================================================================
eval_config = load_yaml(str(PROJECT_ROOT / "configs/evaluation/retrieval.yml"))
# eval_top_k 支持标量或数组；数组时对每个 Top-K 值分别计算并持久化 ASR-R（互不重跑检索）
_eval_top_k_raw = eval_config.get("eval", {}).get("eval_top_k", 10)
eval_top_k_list = list(_eval_top_k_raw) if isinstance(_eval_top_k_raw, (list, tuple)) else [_eval_top_k_raw]
print(f"eval_top_k_list : {eval_top_k_list}")

if max(eval_top_k_list) > retrieval_top_k:
    raise ValueError(
        f"eval_top_k={eval_top_k_list} 超出 APR 数据库检索深度 {retrieval_top_k}，拒绝产生伪 Top-K 指标。"
    )

eval_cwe, configured_eval_mode, effective_eval_mode, report_groups = resolve_eval_groups(
    eval_config.get("eval", {})
)
expected_group_slugs = [slug for slug, _ in report_groups]
eval_retrieval_naive = bool(eval_config.get("eval", {}).get("eval_retrieval_naive", False))
eval_unfinished_jailbreak_target = bool(
    eval_config.get("eval", {}).get("eval_unfinished_jailbreak_target", False)
)

# 检索策略与标签配置（与 Cell 13 保持一致；hybrid 使用前面 Cell 3 解析的真实 alpha_weight）
alpha_label_by_strategy = {
    "sparse": "0",
    "dense": "1",
    "hybrid": str(alpha_weight),
}
display_name_by_strategy = {
    "sparse": "稀疏检索（Sparse）",
    "dense": "密集检索（Dense）",
    "hybrid": "混合检索（Hybrid）",
}
# alpha 标签 → 策略 的反查表（供 from_existing_files 的目录整包解析用）
alpha_to_strategy = {v: k for k, v in alpha_label_by_strategy.items()}

# =============================================================================
# 构建待评估路径集合: {cwe_slug: {label: {strategy: path}}}
# =============================================================================
if MODE == "from_memory":
    # 复用驱动循环产出的 all_output_paths
    print("[MODE] 使用内存中的检索结果（from_memory）")
    try:
        eval_paths = all_output_paths
    except NameError:
        raise RuntimeError(
            "all_output_paths 未定义！请先运行驱动循环 Cell，"
            "或将 MODE 切换为 'from_existing_files' 以从文件加载。"
        )
else:
    print("[MODE] 从已有检索结果文件加载（from_existing_files）")
    results_root = (
        PROJECT_ROOT / "data/query" / division / "test/results"
        / embedder_name / f"{optimizer_name}-{RUN_TIMESTAMP}"
    )

    def glob_strategy_paths(label_dir, cwe_slug):
        """在 label 目录下匹配某 cwe_slug 的新契约结果文件。"""
        found = {}
        for strategy in ["sparse", "dense", "hybrid"]:
            alpha_label = alpha_label_by_strategy[strategy]
            pattern = (
                f"{cwe_slug}_alpha_{alpha_label}"
                f"_target_*_test_*_retrieval_results.json"
            )
            matches = sorted(Path(label_dir).glob(pattern))
            if len(matches) > 1:
                raise RuntimeError(f"{label_dir} 中 {cwe_slug}/{strategy} 匹配到多份结果: {matches}")
            if matches:
                found[strategy] = str(matches[0])
                print(f"  [{cwe_slug}][{strategy}] 找到: {matches[0]}")
        return found

    eval_paths = {}
    if EXISTING_RESULTS_DIR:
        # 手动指定单一 label 目录：形如 ".../{optimizer}/{label}"，整包解析其下所有结果
        base = Path(EXISTING_RESULTS_DIR)
        if base.name == "baseline" or any(base.glob("*_tartget_*")):
            raise ValueError("检测到旧 baseline/tartget 数据契约；请从 Section 3 和检索评估重新生成数据。")
        label = base.name
        fname_re = re.compile(
            r"^(?P<cwe>.+)_alpha_(?P<alpha>[^_]+)"
            r"_target_\d+_test_\d+_retrieval_results\.json$"
        )
        for fp in sorted(base.glob("*_retrieval_results.json")):
            mt = fname_re.match(fp.name)
            if not mt:
                continue
            cwe_slug = mt.group("cwe")
            if cwe_slug not in expected_group_slugs:
                continue
            strategy = alpha_to_strategy.get(mt.group("alpha"))
            if strategy is None:
                continue
            eval_paths.setdefault(cwe_slug, {}).setdefault(label, {})[strategy] = str(fp)
            print(f"  [{cwe_slug}][{label}][{strategy}] 找到: {fp}")
    else:
        # 按三态矩阵重建：与驱动循环使用同一份已验证分组
        group_slugs = expected_group_slugs
        labels = ["naive", "poisoned"] if eval_retrieval_naive else ["poisoned"]
        for cwe_slug in group_slugs:
            for label in labels:
                paths = glob_strategy_paths(results_root / label, cwe_slug)
                if paths:
                    eval_paths.setdefault(cwe_slug, {})[label] = paths

    if not eval_paths:
        raise FileNotFoundError(
            "未找到任何检索结果文件！请确认文件已存在，"
            "或手动设置 EXISTING_RESULTS_DIR 指向正确的 label 结果目录。"
        )

# 无论结果来自内存还是文件，都只保留当前 eval_mode 需要的分组，并恢复配置顺序。
unexpected_group_slugs = [slug for slug in eval_paths if slug not in expected_group_slugs]
if unexpected_group_slugs:
    print(f"[INFO] 按 eval_mode 忽略非目标分组: {unexpected_group_slugs}")
eval_paths = {
    slug: eval_paths[slug]
    for slug in expected_group_slugs
    if slug in eval_paths
}
if not eval_paths:
    raise FileNotFoundError(
        f"当前 eval_mode={configured_eval_mode} 未找到任何目标分组: {expected_group_slugs}"
    )

# 预解析各 (cwe_slug, label) 的 (N, M) 计数（与 Top-K 无关，来自结果文件名，供 overview.md 复用）
# 结构: {cwe_slug: {label: (n_target, n_test)}}
group_counts_from_paths = {}
for cwe_slug, by_label in eval_paths.items():
    group_counts_from_paths.setdefault(cwe_slug, {})
    for label, paths in by_label.items():
        any_path = next(iter(paths.values()), None)
        if any_path:
            n_target, n_test = _parse_counts_from_path(any_path)
            group_counts_from_paths[cwe_slug][label] = (n_target or 0, n_test or 0)


# =============================================================================
# 计算 ASR-R 与 Precision@K 指标 + 日志持久化：逐 Top-K × CWE 分组 × 标签 × 策略
# =============================================================================
# 汇总结构: {top_k: {cwe_slug: {label: {strategy: metrics}}}}
all_metrics_by_k = {}

for k in eval_top_k_list:
    print(f"\n{'='*70}")
    print(f"  评估 Top-K = {k}")
    print(f"{'='*70}")

    all_metrics = {}
    for cwe_slug, by_label in eval_paths.items():
        print(f"\n{'#'*60}")
        print(f"#  CWE 分组: {cwe_slug}")
        print(f"{'#'*60}")
        all_metrics.setdefault(cwe_slug, {})
        for label, paths in by_label.items():
            all_metrics[cwe_slug].setdefault(label, {})
            for strategy, results_path in paths.items():
                alpha_label = alpha_label_by_strategy[strategy]
                display_name = display_name_by_strategy[strategy]
                metrics, report_text = build_retrieval_report(
                    f"[{cwe_slug}][{label}] {display_name} | alpha={alpha_label}",
                    results_path,
                    k,
                    show_detail=SHOW_DETAIL,
                )
                all_metrics[cwe_slug][label][strategy] = metrics
                write_retrieval_log(report_text, cwe_slug, label, alpha_label, results_path, k)

    all_metrics_by_k[k] = all_metrics

    # ── 该 Top-K 下的对比汇总表：同一分组同一策略下 naive 与 poisoned 并列 ──
    print(f"\n{'─'*96}")
    print(f"  对比汇总（Top-{k}）")
    print(f"{'─'*96}")
    print(f"  {'CWE分组':<22}{'检索方式':<14}{'标签':<10}{'ASR-R':>10}{'Precision@K':>14}{'毒文档数':>12}{'命中/总数':>18}")
    print(f"  {'─'*22}{'─'*14}{'─'*10}{'─'*10}{'─'*14}{'─'*12}{'─'*18}")
    for cwe_slug, by_label in all_metrics.items():
        # 收集该分组出现过的所有策略（以策略为外层，便于 naive/poisoned 相邻对比）
        strategies = []
        for label_metrics in by_label.values():
            for strategy in label_metrics:
                if strategy not in strategies:
                    strategies.append(strategy)
        for strategy in strategies:
            for label, label_metrics in by_label.items():
                m = label_metrics.get(strategy)
                if not m:
                    continue
                hit_str = f"{m['hit_count']}/{m['total_queries']}"
                print(
                    f"  {cwe_slug:<22}{display_name_by_strategy[strategy]:<14}{label:<10}"
                    f"{m['asr_r']:>10.4f}{m['precision_at_k']:>14.4f}"
                    f"{m['total_poisoned_count']:>12}{hit_str:>18}"
                )
    print(f"{'─'*96}")

## 8.1 生成 overview.md 汇总报告

汇总 `all_metrics_by_k`（各 Top-K × CWE 分组 × 标签 × 策略的 ASR-R 与未归一化 Precision@K）与 `group_counts_from_paths`（各分组的 N/M 计数），按 `## Top-K = {k}` 分节输出一份 Markdown 总览表，写入：

```
logs/evaluation/retrieval/{embedder}/{optimizer}-{date}/overview.md
```

当同一 Top-K 下 naive 与 poisoned 均存在时，表格按 CWE 分组 × 检索方式并列展示两者的 ASR-R、Precision@K、命中数与差值 Δ；若仅有 poisoned（`eval_retrieval_naive=False`），则退化为单标签的简表。

In [ ]:
def _fmt_pct(x):
    """将 [0,1] 浮点数格式化为 '0.1234 (12.34%)'。"""
    return f"{x:.4f} ({x * 100:.2f}%)"


def build_overview_markdown():
    """基于 all_metrics_by_k + group_counts_from_paths 构建 overview.md 的完整文本。"""
    lines = []
    lines.append("# 检索攻击评估总览（ASR-R + Precision@K）")
    lines.append("")
    lines.append(f"- Embedder  : {embedder_name}")
    lines.append(f"- Optimizer : {optimizer_name}")
    lines.append(f"- Division  : {division}")
    lines.append(f"- Timestamp : {datetime.now().isoformat(timespec='seconds')}")
    lines.append(f"- Poisoned file : {poisoned_file_path}")
    lines.append(f"- Test file     : {test_file_path}")
    lines.append(f"- Eval Top-K    : {eval_top_k_list}")
    lines.append(f"- Eval mode     : {configured_eval_mode}")
    lines.append(f"- Effective eval mode : {effective_eval_mode}")
    lines.append(f"- Eval unfinished jailbreak target : {eval_unfinished_jailbreak_target}")
    injection_stats = globals().get("group_injection_stats", {})
    fallback_summary = {
        f"{cwe_slug}/{label}": stats.get("unfinished_jailbreak_fallback_count", 0)
        for cwe_slug, by_label in injection_stats.items()
        for label, stats in by_label.items()
    }
    lines.append(
        f"- Unfinished jailbreak fallbacks : {fallback_summary if injection_stats else 'unknown'}"
    )
    if any(fallback_summary.values()):
        lines.append(
            "- Result scope : retrieval-only; do not use fallback-containing results for generation attack evaluation"
        )
    lines.append("- Precision@K   : ImportSnare 未归一化口径；每个查询 Top-K 中的平均有毒文档数，不除以 K")
    lines.append("")

    for k in eval_top_k_list:
        by_cwe = all_metrics_by_k.get(k, {})
        lines.append(f"## Top-K = {k}")
        lines.append("")

        # 该 Top-K 下实际出现过的 label 集合（通常为 {naive, poisoned} 或仅 {poisoned}）
        labels_present = set()
        for by_label in by_cwe.values():
            labels_present.update(by_label.keys())
        has_both = labels_present == {"naive", "poisoned"}

        if has_both:
            lines.append(
                "| CWE分组 | 检索方式 | N(naive) | Naive ASR-R | Naive Precision@K | Naive 命中/总数 "
                "| N(poisoned) | Poisoned ASR-R | Poisoned Precision@K | Poisoned 命中/总数 | M(测试样本数) | Δ ASR-R | Δ Precision@K |"
            )
            lines.append("|---|---|---|---|---|---|---|---|---|---|---|---|---|")
        else:
            lines.append(
                "| CWE分组 | 检索方式 | 标签 | N(投毒目标) | ASR-R | Precision@K | 命中/总数 | M(测试样本数) |"
            )
            lines.append("|---|---|---|---|---|---|---|---|")

        for cwe_slug, by_label in by_cwe.items():
            strategies = []
            for label_metrics in by_label.values():
                for strategy in label_metrics:
                    if strategy not in strategies:
                        strategies.append(strategy)
            counts = group_counts_from_paths.get(cwe_slug, {})

            for strategy in strategies:
                display_name = display_name_by_strategy[strategy]
                if has_both:
                    m_base = by_label.get("naive", {}).get(strategy)
                    m_pois = by_label.get("poisoned", {}).get(strategy)
                    n_base, m_base_test = counts.get("naive", (0, 0))
                    n_pois, m_pois_test = counts.get("poisoned", (0, 0))
                    m_test = m_base_test or m_pois_test

                    base_asr = m_base["asr_r"] if m_base else None
                    pois_asr = m_pois["asr_r"] if m_pois else None
                    base_precision = m_base["precision_at_k"] if m_base else None
                    pois_precision = m_pois["precision_at_k"] if m_pois else None
                    base_str = _fmt_pct(base_asr) if m_base else "-"
                    pois_str = _fmt_pct(pois_asr) if m_pois else "-"
                    base_precision_str = f"{base_precision:.4f}" if m_base else "-"
                    pois_precision_str = f"{pois_precision:.4f}" if m_pois else "-"
                    base_hit = f"{m_base['hit_count']}/{m_base['total_queries']}" if m_base else "-"
                    pois_hit = f"{m_pois['hit_count']}/{m_pois['total_queries']}" if m_pois else "-"
                    delta = (
                        f"{(pois_asr - base_asr):+.4f}"
                        if (base_asr is not None and pois_asr is not None)
                        else "-"
                    )
                    precision_delta = (
                        f"{(pois_precision - base_precision):+.4f}"
                        if (base_precision is not None and pois_precision is not None)
                        else "-"
                    )
                    lines.append(
                        f"| {cwe_slug} | {display_name} | {n_base} | {base_str} | {base_precision_str} | {base_hit} "
                        f"| {n_pois} | {pois_str} | {pois_precision_str} | {pois_hit} | {m_test} | {delta} | {precision_delta} |"
                    )
                else:
                    for label, label_metrics in by_label.items():
                        m = label_metrics.get(strategy)
                        if not m:
                            continue
                        n_target, n_test = counts.get(label, (0, 0))
                        lines.append(
                            f"| {cwe_slug} | {display_name} | {label} | {n_target} "
                            f"| {_fmt_pct(m['asr_r'])} | {m['precision_at_k']:.4f} | {m['hit_count']}/{m['total_queries']} | {n_test} |"
                        )
        lines.append("")

    return "\n".join(lines)


overview_path = (
    PROJECT_ROOT / "logs/evaluation/retrieval" / embedder_name
    / f"{optimizer_name}-{RUN_TIMESTAMP}" / "overview.md"
)
overview_path.parent.mkdir(parents=True, exist_ok=True)
overview_path.write_text(build_overview_markdown(), encoding="utf-8")
print(f"[overview] 评估总览已保存: {overview_path}")

## 9. 清理资源

In [ ]:
# 清除本次评估插入的投毒文档，恢复干净的知识库状态
cleared = milvus_client.clear_poisoned_docs()
print(f"已清除投毒文档: {cleared} 条")

milvus_client.close()
print("Milvus 连接已关闭，评估完成。")